# Arabic Text Classification for the AAFAQ Dataset

This notebook develops and compares multiple approaches for classifying Arabic question-answer pairs into the predefined AAFAQ categories.

The experiments include:
- Gaussian Naive Bayes
- Linear SVM
- TF-IDF and Bag-of-Words representations
- AraVec CBOW and Skip-gram embeddings
- AraBERT embeddings
- Fine-tuned AraBERT for sequence classification

Models are evaluated using accuracy, precision, recall, and F1-score.

## 1. Setup and Imports

In [ ]:
!pip install transformers gensim datasets

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from gensim.models import Word2Vec
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import BertTokenizer, BertModel
from datasets import Dataset

## 2. Dataset Loading and Preparation

In [4]:
df = pd.read_csv('/content/AAFAQ_preprocessed.csv')

In [5]:
df.head()

,QuestionText,Clean_Question,Category,Answer,Clean_Answer,question_tokens_split,answer_tokens_split,question_stemmed_snowball,answer_stemmed_snowball,question_lemma_qalsadi,answer_lemma_qalsadi,Lemmatized_Question,Lemmatized_Answer,Stemmed_Question,Stemmed_Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,ايهما افضل الدراسه السابق ام الوقت الحالي,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,الدراسه الوقت الحالي تعتبر افضل بسبب توفر التك...,"['ايهما', 'افضل', 'الدراسه', 'السابق', 'ام', '...","['الدراسه', 'الوقت', 'الحالي', 'تعتبر', 'افضل'...","['ايهم', 'افضل', 'دراسه', 'سابق', 'ام', 'الو',...","['دراسه', 'الو', 'حال', 'تعتبر', 'افضل', 'سبب'...","['وهم', 'فضل', 'الدراسه', 'سابق', 'ام', 'وقت',...","['الدراسه', 'وقت', 'حال', 'اعتبر', 'فضل', 'سبب...",وهم فضل الدراسه سابق ام وقت حال,الدراسه وقت حال اعتبر فضل سبب توفر تكنولوجي ما...,ايهم افضل دراسه سابق ام الو حال,دراسه الو حال تعتبر افضل سبب توفر تكنولوج موار...
1,أليس القطن عماد الثروة في مصر؟,اليس القطن عماد الثروه مصر,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,القطن يعتبر اهم المنتجات الزراعيه مصر ويعد الا...,"['اليس', 'القطن', 'عماد', 'الثروه', 'مصر']","['القطن', 'يعتبر', 'اهم', 'المنتجات', 'الزراعي...","['اليس', 'قطن', 'عماد', 'ثروه', 'مصر']","['قطن', 'يعتبر', 'اهم', 'منتج', 'زراعيه', 'مصر...","['ليس', 'قطن', 'عماد', 'الثروه', 'مصر']","['قطن', 'اعتبر', 'اهم', 'منتج', 'الزراعيه', 'م...",ليس قطن عماد الثروه مصر,قطن اعتبر اهم منتج الزراعيه مصر أعاد الاعمده ا...,اليس قطن عماد ثروه مصر,قطن يعتبر اهم منتج زراعيه مصر يعد اعمده رييسيه...
2,أتصعد الشمس من الشرق؟,اتصعد الشمس الشرق,التعليم,الشمس تصعد من الشرق.,الشمس تصعد الشرق,"['اتصعد', 'الشمس', 'الشرق']","['الشمس', 'تصعد', 'الشرق']","['اتصعد', 'شمس', 'شرق']","['شمس', 'تصعد', 'شرق']","['اتصعد', 'شمس', 'شارق']","['شمس', 'صعد', 'شارق']",اتصعد شمس شارق,شمس صعد شارق,اتصعد شمس شرق,شمس تصعد شرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,اتعرف البكتيريا بانها كاينات حيه دقيقه,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,البكتيريا تعرف بانها كاينات حيه دقيقه,"['اتعرف', 'البكتيريا', 'بانها', 'كاينات', 'حيه...","['البكتيريا', 'تعرف', 'بانها', 'كاينات', 'حيه'...","['اتعرف', 'بكتير', 'بان', 'كاين', 'حيه', 'دقيق']","['بكتير', 'تعرف', 'بان', 'كاين', 'حيه', 'دقيق']","['اتعرف', 'بكتيريا', 'بان', 'كاينات', 'حي', 'د...","['بكتيريا', 'تعرف', 'بان', 'كاينات', 'حي', 'دق...",اتعرف بكتيريا بان كاينات حي دقيق,بكتيريا تعرف بان كاينات حي دقيق,اتعرف بكتير بان كاين حيه دقيق,بكتير تعرف بان كاين حيه دقيق
4,أيتكون الهواء أساساً من النيتروجين؟,ايتكون الهواء اساسا النيتروجين,التعليم,الهواء يتكون أساساً من النيتروجين.,الهواء يتكون اساسا النيتروجين,"['ايتكون', 'الهواء', 'اساسا', 'النيتروجين']","['الهواء', 'يتكون', 'اساسا', 'النيتروجين']","['ايتك', 'هواء', 'اساس', 'نيتروج']","['هواء', 'يتكو', 'اساس', 'نيتروج']","['ايتكون', 'هواء', 'اساسا', 'نيتروجين']","['هواء', 'تك', 'اساسا', 'نيتروجين']",ايتكون هواء اساسا نيتروجين,هواء تك اساسا نيتروجين,ايتك هواء اساس نيتروج,هواء يتكو اساس نيتروج


In [6]:
label_encoder = LabelEncoder()
df['Category_encoded'] = label_encoder.fit_transform(df['Category'])

In [7]:
df['Combined_Text'] = df['Stemmed_Question'].astype(str) + ' ' + df['Stemmed_Answer'].astype(str)

X = df['Combined_Text']
y = df['Category_encoded']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3. Gaussian Naive Bayes Models

### 3.1 Gaussian NB + TF-IDF

In [9]:
tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [10]:
nb_tfidf = GaussianNB()

nb_tfidf.fit(X_train_tfidf, y_train)

GaussianNB()

In [11]:
y_pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)

accuracy_nb_tfidf = accuracy_score(y_test, y_pred_nb_tfidf)
precision_nb_tfidf = precision_score(y_test, y_pred_nb_tfidf, average='weighted')
recall_nb_tfidf = recall_score(y_test, y_pred_nb_tfidf, average='weighted')
f1_nb_tfidf = f1_score(y_test, y_pred_nb_tfidf, average='weighted')

print("Accuracy of Naive Bayes & TF-IDF:", accuracy_nb_tfidf)
print("Precision of Naive Bayes & TF-IDF:", precision_nb_tfidf)
print("Recall of Naive Bayes & TF-IDF:", recall_nb_tfidf)
print("F1 Score of Naive Bayes & TF-IDF:", f1_nb_tfidf)

Accuracy of Naive Bayes & TF-IDF: 0.654690618762475
Precision of Naive Bayes & TF-IDF: 0.6684476496018562
Recall of Naive Bayes & TF-IDF: 0.654690618762475
F1 Score of Naive Bayes & TF-IDF: 0.6516492054234775


### 3.2 Gaussian NB + BoW Unigram

In [12]:
bow_uni = CountVectorizer(ngram_range=(1,1))

X_train_bow_uni = bow_uni.fit_transform(X_train).toarray()
X_test_bow_uni = bow_uni.transform(X_test).toarray()

In [13]:
nb_bow_uni = GaussianNB()

nb_bow_uni.fit(X_train_bow_uni, y_train)

GaussianNB()

In [14]:
y_pred_nb_bow_uni = nb_bow_uni.predict(X_test_bow_uni)

accuracy_nb_bow_uni = accuracy_score(y_test, y_pred_nb_bow_uni)
precision_nb_bow_uni = precision_score(y_test, y_pred_nb_bow_uni, average='weighted')
recall_nb_bow_uni = recall_score(y_test, y_pred_nb_bow_uni, average='weighted')
f1_nb_bow_uni = f1_score(y_test, y_pred_nb_bow_uni, average='weighted')

print("Accuracy of Naive Bayes & BoW Unigram:", accuracy_nb_bow_uni)
print("Precision of Naive Bayes & BoW Unigram:", precision_nb_bow_uni)
print("Recall of Naive Bayes & BoW Unigram:", recall_nb_bow_uni)
print("F1 Score of Naive Bayes & BoW Unigram:", f1_nb_bow_uni)

Accuracy of Naive Bayes & BoW Unigram: 0.6606786427145709
Precision of Naive Bayes & BoW Unigram: 0.673091511185924
Recall of Naive Bayes & BoW Unigram: 0.6606786427145709
F1 Score of Naive Bayes & BoW Unigram: 0.6568558660443469


### 3.3 Gaussian NB + AraVec CBOW

In [15]:
!wget "https://archive.org/download/aravec2.0/tweet_cbow_100.zip"

--2026-06-05 17:41:31--  https://archive.org/download/aravec2.0/tweet_cbow_100.zip
Resolving archive.org (archive.org)... 207.241.224.2
Connecting to archive.org (archive.org)|207.241.224.2|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://dn710903.ca.archive.org/0/items/aravec2.0/tweet_cbow_100.zip [following]
--2026-06-05 17:41:32--  https://dn710903.ca.archive.org/0/items/aravec2.0/tweet_cbow_100.zip
Resolving dn710903.ca.archive.org (dn710903.ca.archive.org)... 204.62.248.114
Connecting to dn710903.ca.archive.org (dn710903.ca.archive.org)|204.62.248.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 253946430 (242M) [application/zip]
Saving to: ‘tweet_cbow_100.zip’

tweet_cbow_100.zip  100%[===================>] 242.18M  19.2MB/s    in 14s     

2026-06-05 17:41:48 (17.4 MB/s) - ‘tweet_cbow_100.zip’ saved [253946430/253946430]



In [16]:
!unzip "tweet_cbow_100.zip"

Archive:  tweet_cbow_100.zip
  inflating: tweets_cbow_100         
  inflating: tweets_cbow_100.trainables.syn1neg.npy  
  inflating: tweets_cbow_100.wv.vectors.npy  


In [17]:
cbow_model = Word2Vec.load("tweets_cbow_100")

In [18]:
cbow_wv = cbow_model.wv

In [19]:
X_train_tokens = X_train.apply(lambda x: x.split())
X_test_tokens = X_test.apply(lambda x: x.split())

In [20]:
question_embeddings = []

for sentence in X_train_tokens:

  vectors =[]

  for word in sentence:
    if word in cbow_wv:
      vectors.append(cbow_wv[word])

  if len(vectors) > 0:
    sentence_vector = np.mean(vectors, axis=0)
  else:
    sentence_vector = np.zeros(100)

  question_embeddings.append(sentence_vector)


In [21]:
X_train_cbow = np.array(question_embeddings)

X_train_cbow.shape

(4007, 100)

In [22]:
test_embeddings = []

for sentence in X_test_tokens:

  vectors =[]

  for word in sentence:
    if word in cbow_wv:
      vectors.append(cbow_wv[word])

  if len(vectors) > 0:
    sentence_vector = np.mean(vectors, axis=0)
  else:
    sentence_vector = np.zeros(100)

  test_embeddings.append(sentence_vector)


In [23]:
X_test_cbow = np.array(test_embeddings)

X_test_cbow.shape

(1002, 100)

In [24]:
nb_cbow = GaussianNB()

nb_cbow.fit(X_train_cbow, y_train)

GaussianNB()

In [25]:
y_pred_nb_cbow = nb_cbow.predict(X_test_cbow)

accuracy_nb_cbow = accuracy_score(y_test, y_pred_nb_cbow)
precision_nb_cbow = precision_score(y_test, y_pred_nb_cbow, average='weighted')
recall_nb_cbow = recall_score(y_test, y_pred_nb_cbow, average='weighted')
f1_nb_cbow = f1_score(y_test, y_pred_nb_cbow, average='weighted')

print("Accuracy of Naive Bayes & AraVec CBOW:", accuracy_nb_cbow)
print("Precision of Naive Bayes & AraVec CBOW:", precision_nb_cbow)
print("Recall of Naive Bayes & AraVec CBOW:", recall_nb_cbow)
print("F1 Score of Naive Bayes & AraVec CBOW:", f1_nb_cbow)

Accuracy of Naive Bayes & AraVec CBOW: 0.5
Precision of Naive Bayes & AraVec CBOW: 0.5181742780501373
Recall of Naive Bayes & AraVec CBOW: 0.5
F1 Score of Naive Bayes & AraVec CBOW: 0.48955780341852567


### 3.4 Gaussian NB + AraVec Skip-gram

In [26]:
!wget "https://archive.org/download/aravec2.0/tweets_sg_100.zip"

--2026-06-05 17:41:58--  https://archive.org/download/aravec2.0/tweets_sg_100.zip
Resolving archive.org (archive.org)... 207.241.224.2
Connecting to archive.org (archive.org)|207.241.224.2|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://dn710903.ca.archive.org/0/items/aravec2.0/tweets_sg_100.zip [following]
--2026-06-05 17:41:59--  https://dn710903.ca.archive.org/0/items/aravec2.0/tweets_sg_100.zip
Resolving dn710903.ca.archive.org (dn710903.ca.archive.org)... 204.62.248.114
Connecting to dn710903.ca.archive.org (dn710903.ca.archive.org)|204.62.248.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 254154396 (242M) [application/zip]
Saving to: ‘tweets_sg_100.zip’

tweets_sg_100.zip   100%[===================>] 242.38M  10.6MB/s    in 17s     

2026-06-05 17:42:16 (14.7 MB/s) - ‘tweets_sg_100.zip’ saved [254154396/254154396]



In [27]:
!unzip "tweets_sg_100.zip"

Archive:  tweets_sg_100.zip
  inflating: tweets_sg_100.trainables.syn1neg.npy  
  inflating: tweets_sg_100.wv.vectors.npy  
  inflating: tweets_sg_100           


In [28]:
sg_model = Word2Vec.load("tweets_sg_100")

In [29]:
sg_wv = sg_model.wv

In [30]:
skipgram_train_embeddings = []

for sentence in X_train_tokens:

  vectors =[]

  for word in sentence:
    if word in sg_wv:
      vectors.append(sg_wv[word])

  if len(vectors) > 0:
    sentence_vector = np.mean(vectors, axis=0)
  else:
    sentence_vector = np.zeros(100)

  skipgram_train_embeddings.append(sentence_vector)

X_train_sg = np.array(skipgram_train_embeddings)

In [31]:
skipgram_test_embeddings = []

for sentence in X_test_tokens:

  vectors =[]

  for word in sentence:
    if word in sg_wv:
      vectors.append(sg_wv[word])

  if len(vectors) > 0:
    sentence_vector = np.mean(vectors, axis=0)

  else:
    sentence_vector = np.zeros(100)

  skipgram_test_embeddings.append(sentence_vector)

X_test_sg = np.array(skipgram_test_embeddings)

In [32]:
nb_sg = GaussianNB()

nb_sg.fit(X_train_sg, y_train)

GaussianNB()

In [33]:
y_pred_nb_sg = nb_sg.predict(X_test_sg)

accuracy_nb_sg = accuracy_score(y_test, y_pred_nb_sg)
precision_nb_sg = precision_score(y_test, y_pred_nb_sg, average='weighted')
recall_nb_sg = recall_score(y_test, y_pred_nb_sg, average='weighted')
f1_nb_sg = f1_score(y_test, y_pred_nb_sg, average='weighted')

print("Accuracy of Naive Bayes & AraVec Skip-gram:", accuracy_nb_sg)
print("Precision of Naive Bayes & AraVec Skip-gram:", precision_nb_sg)
print("Recall of Naive Bayes & AraVec Skip-gram:", recall_nb_sg)
print("F1 Score of Naive Bayes & AraVec Skip-gram:", f1_nb_sg)

Accuracy of Naive Bayes & AraVec Skip-gram: 0.5289421157684631
Precision of Naive Bayes & AraVec Skip-gram: 0.5364717078712751
Recall of Naive Bayes & AraVec Skip-gram: 0.5289421157684631
F1 Score of Naive Bayes & AraVec Skip-gram: 0.5154007719431806


### 3.5 Gaussian NB + AraBERT Embeddings

In [34]:
X_bert = df['QuestionText'].astype(str) + ' ' + df['Answer'].astype(str)
y_bert = df['Category_encoded']

In [35]:
X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    X_bert, y_bert, test_size=0.2, random_state=42)

In [36]:
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")
model_bert = AutoModel.from_pretrained("aubmindlab/bert-base-arabertv02")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_bert = model_bert.to(device)

def get_bert_embeddings(texts, batch_size=32):
  all_embeddings = []
  for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i + batch_size]
    inputs = tokenizer(
        batch_texts.tolist(),
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
      outputs = model_bert(**inputs)

    embeddings = outputs.last_hidden_state.mean(dim=1)
    all_embeddings.append(embeddings.cpu().numpy())
  return np.concatenate(all_embeddings, axis=0)


In [40]:
X_train_bert_embeddings = get_bert_embeddings(X_train_bert)
X_test_bert_embeddings = get_bert_embeddings(X_test_bert)

In [41]:
nb_bert = GaussianNB()

nb_bert.fit(X_train_bert_embeddings, y_train_bert)

GaussianNB()

In [42]:
y_pred_nb_bert = nb_bert.predict(X_test_bert_embeddings)

accuracy_nb_bert = accuracy_score(y_test_bert, y_pred_nb_bert)
precision_nb_bert = precision_score(y_test_bert, y_pred_nb_bert, average='weighted')
recall_nb_bert = recall_score(y_test_bert, y_pred_nb_bert, average='weighted')
f1_nb_bert = f1_score(y_test_bert, y_pred_nb_bert, average='weighted')

print("Accuracy of Naive Bayes & BERT:", accuracy_nb_bert)
print("Precision of Naive Bayes & BERT:", precision_nb_bert)
print("Recall of Naive Bayes & BERT:", recall_nb_bert)
print("F1 Score of Naive Bayes & BERT:", f1_nb_bert)

Accuracy of Naive Bayes & BERT: 0.6696606786427146
Precision of Naive Bayes & BERT: 0.6934665598463856
Recall of Naive Bayes & BERT: 0.6696606786427146
F1 Score of Naive Bayes & BERT: 0.6711750834814961


### 3.6 Gaussian Naive Bayes Model Comparison

In [43]:
nb_results = pd.DataFrame({
    'Technique': ['Naive Bayes & TF-IDF', 'Naive Bayes & BoW Unigram','Naive Bayes & AraVec CBOW', 'Naive Bayes & AraVec Skip-gram','Naive Bayes & AraBERT Embeddings'],
    'Accuracy': [accuracy_nb_tfidf, accuracy_nb_bow_uni, accuracy_nb_cbow, accuracy_nb_sg, accuracy_nb_bert],
    'Precision': [precision_nb_tfidf, precision_nb_bow_uni, precision_nb_cbow, precision_nb_sg, precision_nb_bert],
    'Recall': [recall_nb_tfidf, recall_nb_bow_uni, recall_nb_cbow,recall_nb_sg, recall_nb_bert],
    'F1 Score': [f1_nb_tfidf, f1_nb_bow_uni, f1_nb_cbow, f1_nb_sg, f1_nb_bert]
})

print('Naive Bayes results')
nb_results

Naive Bayes results


,Technique,Accuracy,Precision,Recall,F1 Score
0,Naive Bayes & TF-IDF,0.654691,0.668448,0.654691,0.651649
1,Naive Bayes & BoW Unigram,0.660679,0.673092,0.660679,0.656856
2,Naive Bayes & AraVec CBOW,0.500000,0.518174,0.500000,0.489558
3,Naive Bayes & AraVec Skip-gram,0.528942,0.536472,0.528942,0.515401
4,Naive Bayes & AraBERT Embeddings,0.669661,0.693467,0.669661,0.671175


## 4. Support Vector Machine Models

### 4.1 SVM + TF-IDF

In [44]:
svm_tfidf = LinearSVC()

svm_tfidf.fit(X_train_tfidf, y_train)

LinearSVC()

In [45]:
y_pred_svm_tfidf = svm_tfidf.predict(X_test_tfidf)

accuracy_svm_tfidf = accuracy_score(y_test, y_pred_svm_tfidf)
precision_svm_tfidf = precision_score(y_test, y_pred_svm_tfidf, average='weighted')
recall_svm_tfidf = recall_score(y_test, y_pred_svm_tfidf, average='weighted')
f1_svm_tfidf = f1_score(y_test, y_pred_svm_tfidf, average='weighted')

print("Accuracy of SVM & TF-IDF:", accuracy_svm_tfidf)
print("Precision of SVM & TF-IDF:", precision_svm_tfidf)
print("Recall of SVM & TF-IDF:", recall_svm_tfidf)
print("F1 Score of SVM & TF-IDF:", f1_svm_tfidf)

Accuracy of SVM & TF-IDF: 0.783433133732535
Precision of SVM & TF-IDF: 0.7872769808695502
Recall of SVM & TF-IDF: 0.783433133732535
F1 Score of SVM & TF-IDF: 0.782871402035166


### 4.2 SVM + BoW Unigram

In [46]:
svm_bow_uni = LinearSVC()

svm_bow_uni.fit(X_train_bow_uni, y_train)

LinearSVC()

In [47]:
y_pred_svm_bow_uni = svm_bow_uni.predict(X_test_bow_uni)

accuracy_svm_bow_uni = accuracy_score(y_test, y_pred_svm_bow_uni)
precision_svm_bow_uni = precision_score(y_test, y_pred_svm_bow_uni, average='weighted')
recall_svm_bow_uni = recall_score(y_test, y_pred_svm_bow_uni, average='weighted')
f1_svm_bow_uni = f1_score(y_test, y_pred_svm_bow_uni, average='weighted')

print("Accuracy of SVM & BoW Unigram:", accuracy_svm_bow_uni)
print("Precision of SVM & BoW Unigram:", precision_svm_bow_uni)
print("Recall of SVM & BoW Unigram:", recall_svm_bow_uni)
print("F1 Score of SVM & BoW Unigram:", f1_svm_bow_uni)

Accuracy of SVM & BoW Unigram: 0.7564870259481038
Precision of SVM & BoW Unigram: 0.7638830055825003
Recall of SVM & BoW Unigram: 0.7564870259481038
F1 Score of SVM & BoW Unigram: 0.7579004914503713


### 4.3 SVM + AraVec CBOW

In [48]:
svm_cbow = LinearSVC()

svm_cbow.fit(X_train_cbow, y_train)

LinearSVC()

In [49]:
y_pred_svm_cbow = svm_cbow.predict(X_test_cbow)

accuracy_svm_cbow = accuracy_score(y_test, y_pred_svm_cbow)
precision_svm_cbow = precision_score(y_test, y_pred_svm_cbow, average='weighted')
recall_svm_cbow = recall_score(y_test, y_pred_svm_cbow, average='weighted')
f1_svm_cbow = f1_score(y_test, y_pred_svm_cbow, average='weighted')

print("Accuracy of SVM & AraVec CBOW:", accuracy_svm_cbow)
print("Precision of SVM & AraVec CBOW:", precision_svm_cbow)
print("Recall of SVM & AraVec CBOW:", recall_svm_cbow)
print("F1 Score of SVM & AraVec CBOW:", f1_svm_cbow)

Accuracy of SVM & AraVec CBOW: 0.6067864271457086
Precision of SVM & AraVec CBOW: 0.6143454024598448
Recall of SVM & AraVec CBOW: 0.6067864271457086
F1 Score of SVM & AraVec CBOW: 0.6059956960240295


### 4.4 SVM + AraVec Skip-gram

In [50]:
svm_sg = LinearSVC()

svm_sg.fit(X_train_sg, y_train)

LinearSVC()

In [51]:
y_pred_svm_sg = svm_sg.predict(X_test_sg)

accuracy_svm_sg = accuracy_score(y_test, y_pred_svm_sg)
precision_svm_sg = precision_score(y_test, y_pred_svm_sg, average='weighted')
recall_svm_sg = recall_score(y_test, y_pred_svm_sg, average='weighted')
f1_svm_sg = f1_score(y_test, y_pred_svm_sg, average='weighted')

print("Accuracy of SVM & AraVec Skip-gram:", accuracy_svm_sg)
print("Precision of SVM & AraVec Skip-gram:", precision_svm_sg)
print("Recall of SVM & AraVec Skip-gram:", recall_svm_sg)
print("F1 Score of SVM & AraVec Skip-gram:", f1_svm_sg)

Accuracy of SVM & AraVec Skip-gram: 0.6387225548902196
Precision of SVM & AraVec Skip-gram: 0.6441636556024916
Recall of SVM & AraVec Skip-gram: 0.6387225548902196
F1 Score of SVM & AraVec Skip-gram: 0.6365257598078187


### 4.5 SVM + AraBERT Embeddings

In [52]:
svm_bert = LinearSVC()

svm_bert.fit(X_train_bert_embeddings, y_train_bert)

LinearSVC()

In [53]:
y_pred_svm_bert = svm_bert.predict(X_test_bert_embeddings)

accuracy_svm_bert = accuracy_score(y_test_bert, y_pred_svm_bert)
precision_svm_bert = precision_score(y_test_bert, y_pred_svm_bert, average='weighted')
recall_svm_bert = recall_score(y_test_bert, y_pred_svm_bert, average='weighted')
f1_svm_bert = f1_score(y_test_bert, y_pred_svm_bert, average='weighted')

print("Accuracy of SVM & BERT:", accuracy_svm_bert)
print("Precision of SVM & BERT:", precision_svm_bert)
print("Recall of SVM & BERT:", recall_svm_bert)
print("F1 Score of SVM & BERT:", f1_svm_bert)

Accuracy of SVM & BERT: 0.7844311377245509
Precision of SVM & BERT: 0.7954586603098844
Recall of SVM & BERT: 0.7844311377245509
F1 Score of SVM & BERT: 0.7850299108893288


### 4.6 SVM Model Comparison

In [54]:
svm_results = pd.DataFrame({
    'Technique': ['SVM & TF-IDF', 'SVM & BoW Unigram', 'SVM & AraVec CBOW', 'SVM & AraVec Skip-gram', 'SVM & AraBERT Embeddings'],
    'Accuracy': [accuracy_svm_tfidf, accuracy_svm_bow_uni, accuracy_svm_cbow, accuracy_svm_sg, accuracy_svm_bert],
    'Precision': [precision_svm_tfidf, precision_svm_bow_uni, precision_svm_cbow, precision_svm_sg, precision_svm_bert],
    'Recall': [recall_svm_tfidf, recall_svm_bow_uni, recall_svm_cbow, recall_svm_sg, recall_svm_bert],
    'F1 Score': [f1_svm_tfidf, f1_svm_bow_uni, f1_svm_cbow, f1_svm_sg, f1_svm_bert]
})

print("SVM results")
svm_results

SVM results


,Technique,Accuracy,Precision,Recall,F1 Score
0,SVM & TF-IDF,0.783433,0.787277,0.783433,0.782871
1,SVM & BoW Unigram,0.756487,0.763883,0.756487,0.757900
2,SVM & AraVec CBOW,0.606786,0.614345,0.606786,0.605996
3,SVM & AraVec Skip-gram,0.638723,0.644164,0.638723,0.636526
4,SVM & AraBERT Embeddings,0.784431,0.795459,0.784431,0.785030


## 5. Fine-Tuned AraBERT Classification

In [55]:
X_bert = df['QuestionText'].astype(str) + ' ' + df['Answer'].astype(str)
y_bert = df['Category_encoded']

In [56]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    X_bert, y_bert, test_size=0.2, random_state=42)

In [57]:
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")

In [58]:
def tokenize_function(texts, tokenizer):
  return tokenizer(
      texts.tolist(),
      padding=True,
      truncation=True,
      max_length=128
  )

In [59]:
train_encodings = tokenize_function(train_texts, tokenizer)
test_encodings = tokenize_function(test_texts, tokenizer)

In [60]:
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels.tolist()
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': test_labels.tolist()
})

In [61]:
num_labels = df['Category_encoded'].nunique()

bert_model = AutoModelForSequenceClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv02",
    num_labels=num_labels
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [62]:
training_args = TrainingArguments(
    output_dir='./arabert_results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    optim="adamw_torch"
)

In [63]:
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [64]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.368439,0.779772
2,0.523426,0.548440
3,0.288968,0.547433


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1503, training_loss=0.7257681122162464, metrics={'train_runtime': 356.2211, 'train_samples_per_second': 33.746, 'train_steps_per_second': 4.219, 'total_flos': 790820991717120.0, 'train_loss': 0.7257681122162464, 'epoch': 3.0})

In [65]:
predictions = trainer.predict(test_dataset)

preds = predictions.predictions.argmax(axis=-1)

accuracy_bert = accuracy_score(test_labels, preds)
precision_bert = precision_score(test_labels, preds, average='weighted')
recall_bert = recall_score(test_labels, preds, average='weighted')
f1_bert = f1_score(test_labels, preds, average='weighted')

print("Accuracy of BERT:", accuracy_bert)
print("Precision of BERT:", precision_bert)
print("Recall of BERT:", recall_bert)
print("F1 Score of BERT:", f1_bert)

Accuracy of BERT: 0.8592814371257484
Precision of BERT: 0.8655299351697178
Recall of BERT: 0.8592814371257484
F1 Score of BERT: 0.8593057792183686


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [66]:
bert_results = pd.DataFrame({
    'Technique': ['BERT'],
    'Accuracy': [accuracy_bert],
    'Precision': [precision_bert],
    'Recall': [recall_bert],
    'F1 Score': [f1_bert]
})

## 6. Final Model Comparison

In [67]:
all_results = pd.concat([nb_results, svm_results, bert_results], ignore_index=True)

print("Final Classification Results")
all_results

Final Classification Results


,Technique,Accuracy,Precision,Recall,F1 Score
0,Naive Bayes & TF-IDF,0.654691,0.668448,0.654691,0.651649
1,Naive Bayes & BoW Unigram,0.660679,0.673092,0.660679,0.656856
2,Naive Bayes & AraVec CBOW,0.500000,0.518174,0.500000,0.489558
3,Naive Bayes & AraVec Skip-gram,0.528942,0.536472,0.528942,0.515401
4,Naive Bayes & AraBERT Embeddings,0.669661,0.693467,0.669661,0.671175
5,SVM & TF-IDF,0.783433,0.787277,0.783433,0.782871
6,SVM & BoW Unigram,0.756487,0.763883,0.756487,0.757900
7,SVM & AraVec CBOW,0.606786,0.614345,0.606786,0.605996
8,SVM & AraVec Skip-gram,0.638723,0.644164,0.638723,0.636526
9,SVM & AraBERT Embeddings,0.784431,0.795459,0.784431,0.785030


## 7. Selected Model and Export

In [68]:
#Bert was the best model for classification
bert_model.save_pretrained("best_classification_bert")
tokenizer.save_pretrained("best_classification_bert")

import pickle

with open("label_encoder.pkl", "wb") as f:
  pickle.dump(label_encoder, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [69]:
!zip -r best_classification_bert.zip best_classification_bert

  adding: best_classification_bert/ (stored 0%)
  adding: best_classification_bert/tokenizer.json (deflated 74%)
  adding: best_classification_bert/tokenizer_config.json (deflated 44%)
  adding: best_classification_bert/model.safetensors (deflated 7%)
  adding: best_classification_bert/config.json (deflated 62%)
